<a href="https://colab.research.google.com/github/AbdulHanoos/Drive_Guard/blob/main/Driver_Distraction_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()

Saving kaggle .json to kaggle .json


{'kaggle .json': b'{"username":"akahanoos","key":"ed9ffbc8779815cf9c121113d3eb2859"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp "kaggle .json" ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle .json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
!ls ~/.kaggle

In [ ]:
!kaggle competitions download -c state-farm-distracted-driver-detection -p /content/distracted_driver_data

100% 4.00G/4.00G [00:40<00:00, 106MB/s]



In [ ]:
!unzip -q /content/distracted_driver_data/state-farm-distracted-driver-detection.zip -d /content/distracted_driver_data/state-farm-distracted-driver-detection

In [ ]:
import os

train_path = "/content/distracted_driver_data/state-farm-distracted-driver-detection/imgs/train"

classes = sorted([
    cls for cls in os.listdir(train_path)
    if os.path.isdir(os.path.join(train_path, cls))
])

print("Classes found:", classes)
print("Number of classes:", len(classes))

for cls in classes:
    class_path = os.path.join(train_path, cls)
    image_count = len([
        img for img in os.listdir(class_path)
        if img.lower().endswith((".jpg", ".jpeg", ".png"))
    ])
    print(cls, ":", image_count)

FileNotFoundError: [Errno 2] No such file or directory: '/content/distracted_driver_data/state-farm-distracted-driver-detection/imgs/train'

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from imutils import paths
import random
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow.keras.utils import img_to_array


In [ ]:
imagePaths = list(paths.list_images("/content/distracted_driver_data/state-farm-distracted-driver-detection/imgs/train"))
classnames = {
    "c0": "normal driving",
    "c1": "texting - right",
    "c2": "talking on the phone - right",
    "c3": "texting - left",
    "c4": "talking on the phone - left",       # Fixed: was "right"
    "c5": "operating the radio",               # Fixed: typo "ratio"
    "c6": "drinking",
    "c7": "reaching behind",
    "c8": "hair and makeup",
    "c9": "talking to passenger"
}

def display_images(image_paths, title, cols=5, num_images=25):  # Set cols and add num_images
    random.shuffle(image_paths)  # Shuffle the image paths randomly
    num_images = min(len(image_paths), num_images)  # Limit the number of images to display
    rows = (num_images + cols - 1) // cols

    plt.figure(figsize=(12, 12))  # Adjust figure size to accommodate more rows
    plt.suptitle(title)

    for i, image_path in enumerate(image_paths[:num_images]):
        plt.subplot(rows, cols, i + 1)
        img = Image.open(image_path)
        plt.imshow(img,cmap = "gray")
        label = classnames.get(os.path.basename(os.path.dirname(image_path)))
        plt.title(label)
        plt.axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# Display a random set of images in a 5x5 grid
display_images(imagePaths, title="Random Sample of Images from Different Classes", num_images=25)


NameError: name 'paths' is not defined

In [ ]:
import os
import random
import matplotlib.pyplot as plt

# Folder containing all classes
data_folder = "/content/distracted_driver_data/state-farm-distracted-driver-detection/imgs/train"

# Number of images needed for train and validation
train_count = 500
val_count = 100

# Dictionary to store train and validation paths
train_paths = {cls: [] for cls in os.listdir(data_folder)}
val_paths = {cls: [] for cls in os.listdir(data_folder)}

# Iterate through each class folder
for cls in train_paths.keys():
    class_folder = os.path.join(data_folder, cls)
    if os.path.isdir(class_folder):
        # Get all image paths in the class folder
        images = [os.path.join(class_folder, img) for img in os.listdir(class_folder) if img.endswith(('.png', '.jpg', '.jpeg'))]
        # Shuffle the images to randomize the selection
        random.shuffle(images)
        # Split into train and validation
        train_paths[cls] = images[:train_count]
        val_paths[cls] = images[train_count:train_count + val_count]

# Count images per class for train and validation
train_counts = {cls: len(paths) for cls, paths in train_paths.items()}
val_counts = {cls: len(paths) for cls, paths in val_paths.items()}

# Visualization: Bar plot for training image counts
plt.figure(figsize=(10, 6))
plt.bar(train_counts.keys(), train_counts.values(), color='green', alpha=0.7)
plt.xlabel("Class Names", fontsize=12)
plt.ylabel("Number of Training Images", fontsize=12)
plt.title("Number of Training Images Per Class", fontsize=14)
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.tight_layout()
plt.show()

# Visualization: Bar plot for validation image counts
plt.figure(figsize=(10, 6))
plt.bar(val_counts.keys(), val_counts.values(), color='blue', alpha=0.7)
plt.xlabel("Class Names", fontsize=12)
plt.ylabel("Number of Validation Images", fontsize=12)
plt.title("Number of Validation Images Per Class", fontsize=14)
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/distracted_driver_data/state-farm-distracted-driver-detection/imgs/train'

In [ ]:
import cv2
import os
import numpy as np

class SimpleDatasetLoader:
  def __init__(self, preprocessors=None):
    self.preprocessors = preprocessors
    if self.preprocessors is None:
      self.preprocessors = []

  def load(self, imagePaths, verbose =- 1):
  # initialize the list of features and labels
    data = []
    labels = []

    for (i, imagePath) in enumerate(imagePaths):
      image = cv2.imread(imagePath)
      label = imagePath.split(os.path.sep)[-2]

# check to see if our preprocessors are not None
      if self.preprocessors is not None:
        for p in self.preprocessors:
          image = p.preprocess(image)

# treat our processed image as a "feature vector"
# by updating the data list followed by the labels
      data.append(image)
      labels.append(label)
      #-show.an update-every 'verbose" . images
      if verbose > 0 and i > 0 and (i + 1) % verbose == 0:
        print("[INFO] processed {}/{}".format(i +1,
          len(imagePaths)))

# return a tuple of the data and labels
    return (np.array(data), np.array(labels))

In [ ]:
import cv2
class SimplePreprocessor:
    def __init__(self, width, height, inter=cv2.INTER_AREA):
        self.width = width
        self.height = height
        self.inter = inter

    def preprocess(self, image):
      return cv2.resize(image, (self.width, self.height),
                        interpolation=self.inter)



In [ ]:
from tensorflow.keras.preprocessing.image import img_to_array

class ImageToArrayPreprocessor:
    def __init__(self, dataFormat=None):
        self.dataFormat = dataFormat

    def preprocess(self, image):
        return img_to_array(image, data_format=self.dataFormat)

In [ ]:
 # initialize the image preprocessors
sp = SimplePreprocessor(128,128)
iap = ImageToArrayPreprocessor()

In [ ]:

# Combine all training image paths into a single list
all_train_paths = [path for paths in train_paths.values() for path in paths]

# Combine all validation image paths into a single list (optional)
all_val_paths = [path for paths in val_paths.values() for path in paths]

# Output the size of the lists
print(f"Total training images: {len(all_train_paths)}")
print(f"Total validation images: {len(all_val_paths)}")

Total training images: 5000
Total validation images: 1000


In [ ]:
iap = ImageToArrayPreprocessor()
sdl = SimpleDatasetLoader(preprocessors=[sp, iap])
(train_data, train_labels) = sdl.load(all_train_paths, verbose=1000)

[INFO] processed 1000/5000
[INFO] processed 2000/5000
[INFO] processed 3000/5000
[INFO] processed 4000/5000
[INFO] processed 5000/5000


In [ ]:
(val_data, val_labels) = sdl.load(all_val_paths, verbose=500)

[INFO] processed 500/1000
[INFO] processed 1000/1000


In [ ]:
print(type(train_data))
print(train_data.shape)

print(type(val_data))
print(val_data.shape)

<class 'numpy.ndarray'>
(5000, 128, 128, 3)
<class 'numpy.ndarray'>
(1000, 128, 128, 3)


In [ ]:
[ ] # Data Normalization for all-train and validatoin all.image
train_scaled_data = train_data.astype('float32')/255
val_scaled_data = val_data.astype('float32')/255

In [ ]:


print(train_scaled_data.shape)
print(val_scaled_data.shape)
print(train_scaled_data.min(), train_scaled_data.max())
print(val_scaled_data.min(), val_scaled_data.max())

(5000, 128, 128, 3)
(1000, 128, 128, 3)
0.0 1.0
0.0 1.0


In [ ]:
import pandas as pd
train_val_data = {'total data':[len(train_scaled_data),len(val_scaled_data),len(train_labels),len(val_labels)]}
pd.DataFrame.from_dict(train_val_data, orient='index',columns=["train_scaled_data","train_labels","val_scaled_data","val_labels"])

,train_scaled_data,train_labels,val_scaled_data,val_labels
total data,5000,1000,5000,1000


In [ ]:
# Calculate the number of images for training and testing
import matplotlib.pyplot as plt
num_train_images = len(train_scaled_data)
num_test_images = len(val_scaled_data)
# Create a bar plot
plt.figure(figsize=(8, 5))
plt.bar(['Training', 'Testing'], [num_train_images, num_test_images], color=['blue', 'orange'])
plt.xlabel('Dataset')
plt.ylabel('Number of Images')
plt.title('Number of Images in Training and Testing Datasets')
plt.show()

NameError: name 'train_scaled_data' is not defined

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Convert labels to DataFrames
train_df = pd.DataFrame({"Class": train_labels})
val_df = pd.DataFrame({"Class": val_labels})

plt.figure(figsize=(12, 6))

sns.countplot(
    x="Class",
    data=train_df,
    color="red",
    alpha=0.6,
    label="Training Set"
)

sns.countplot(
    x="Class",
    data=val_df,
    color="cyan",
    alpha=0.6,
    label="Validation Set"
)

plt.title("Class Distribution in Training and Validation Sets")
plt.xlabel("Class")
plt.ylabel("Count")
plt.legend()
plt.show()

NameError: name 'train_labels' is not defined

In [ ]:
from sklearn.preprocessing import LabelBinarizer
y_train_label = LabelBinarizer().fit_transform(train_labels)
y_test_label = LabelBinarizer().fit_transform(val_labels)

In [ ]:
y_train_label

array([[0, 1, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0]])

In [ ]:
import os
import h5py
import math
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model, load_model
from tensorflow.keras import layers
from tensorflow.keras.layers import Input, Add, Dense, Activation, ZeroPadding2D, BatchNormalization, Flatten, Conv2D, AveragePooling2D, MaxPooling2D, GlobalMaxPooling2D
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.imagenet_utils import preprocess_input
from IPython.display import SVG
from tensorflow.keras.utils import plot_model
from tensorflow.keras.initializers import glorot_uniform
import scipy.misc
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow
from tensorflow.keras.optimizers import Adam
%matplotlib inline
import tensorflow.keras.backend as K
K.set_image_data_format('channels_last')
TF_USE_LEGACY_KERAS=True

/tmp/ipykernel_8563/1423892052.py:15: DeprecationWarning: scipy.misc is deprecated and will be removed in 2.0.0
  import scipy.misc


In [ ]:
def identity_block(X, f, filters, stage, block):
# defining base name for block
    conv_base_name = 'res' + str(stage) + block + '_'
    bn_base_name = 'bn' + str(stage) + block + '_'

# retrieve number of filters in each layer of main
# NOTE: f3 must be equal to n_C. That way dimensio
    f1, f2, f3 = filters

# Batch normalization must be performed on the 'ch
    bn_axis = 3

# save input for "addition" to last layer output;
    X_skip_connection = X
    X = Conv2D(filters= f1, kernel_size = (1,1), strides = (1,1), padding='valid', name=conv_base_name+'first_component', kernel_initializer = glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=bn_axis, name=bn_base_name+'first_component')(X)
    X = Activation('relu')(X)

# Second component/layer of main path
    X = Conv2D(filters= f2, kernel_size = (f,f), strides = (1,1), padding='same', name=conv_base_name+'second_component', kernel_initializer = glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=bn_axis, name=bn_base_name+'second_component')(X)
    X = Activation('relu')(X)

# Third component/layer of main path
    X = Conv2D(filters= f3, kernel_size = (1,1), strides = (1,1), padding='valid', name=conv_base_name+'third_component', kernel_initializer = glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=bn_axis, name=bn_base_name+'third_component')(X)

# "Addition step" - skip-connection value merges with main path
# NOTE: both values have same dimensions at this point, so no operation is required to match dimensions
    X = Add()([X, X_skip_connection])
    X = Activation('relu')(X)

    return X

In [ ]:
def convolutional_block(X, f, filters, stage, block, s = 2):
    conv_base_name = 'res' + str(stage) + block +'_'
    bn_base_name = 'bn' + str(stage) + block + '_'

# retrieve number of filters in each layer of main path
    f1, f2, f3 = filters

# Batch normalization must be performed on the 'channels' axis for input. It is 3, for our case
    bn_axis = 3

# save input for "addition" to last layer output; step in skip-connection
    X_skip_connection = X

##### MAIN PATH #####
# First component of main path
    X = Conv2D(f1,(1, 1), strides = (s,s), padding = 'valid', name = conv_base_name + 'first_component', kernel_initializer = glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis = bn_axis, name = bn_base_name + 'first_component')(X)
    X = Activation('relu') (X)

# Second component of main path
    X = Conv2D(f2, kernel_size = (f, f), strides = (1,1), padding ='same', name= conv_base_name+'second_component', kernel_initializer = glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis = bn_axis, name = bn_base_name + 'second_component')(X)
    X = Activation('relu') (X)

# Third component of main path
    X = Conv2D(f3, kernel_size  = (1, 1), strides = (1,1), padding = 'valid', name = conv_base_name + 'third_component', kernel_initializer = glorot_uniform(seed= 0))(X)
    X = BatchNormalization(axis = bn_axis, name = bn_base_name + 'third_component')(X)

##### Convolve skip-connection value to match its dimensions to third layer output's dimensions ####
    X_skip_connection = Conv2D(f3, (1, 1), strides = (s,s), padding = 'valid', name = conv_base_name + 'merge', kernel_initializer = glorot_uniform(seed=0))(X_skip_connection)
    X_skip_connection = BatchNormalization(axis = 3, name = bn_base_name + 'merge')(X_skip_connection)

# "Addition step"
# NOTE: both values have same dimensions at this point
    X = Add()([X, X_skip_connection])
    X = Activation('relu') (X)

    return X

In [ ]:
def ResNet50(input_shape,classes ):

# plug in input_shape to define the input tensor
    X_input = Input(input_shape)

# Zero-Padding : pads the input with a pad of (3,3)
    X = ZeroPadding2D((3, 3))(X_input)

# Stage 1
    X = Conv2D(64, (7, 7), strides = (2, 2), name = 'conv_1', kernel_initializer = glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis = 3, name = 'bn_1')(X)
    X = Activation('relu') (X)
    X = MaxPooling2D((3, 3), strides=(2, 2))(X)

# NOTE: dimensions of filters that are passed to identity block are such that final layer output
# in identity block mathces the original input to the block
# blocks in each stage are alphabetically sequenced

# Stage 2
    X = convolutional_block(X, f = 3, filters = [64, 64, 256], stage = 2, block='a', s = 1)
    X = identity_block(X, 3, [64, 64, 256], stage=2, block='b')
    X = identity_block(X, 3, [64, 64, 256], stage=2, block='c')

# Stage 3
    X = convolutional_block(X, f=3, filters=[128, 128, 512], stage=3, block='a', s=2)
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='b')
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='c')
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='d')

# Stage 4
    X = convolutional_block(X, f=3, filters=[256, 256, 1024], stage=4, block='a', s=2)
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='b')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='c')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='d')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='e')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='f')

# Stage 5
    X = convolutional_block(X, f=3, filters=[512, 512, 2048], stage=5, block='a', s=2)
    X = identity_block(X, 3, [512, 512, 2048], stage=5, block='b')
    X = identity_block(X, 3, [512, 512, 2048], stage=5, block='c')

# Average Pooling
    X = AveragePooling2D((2, 2), name='avg_pool')(X)

# output layer
    X = Flatten()(X)
    X = Dense(512, activation='relu', kernel_initializer=glorot_uniform(seed=0))(X)
    X = Dense(256, activation='relu', kernel_initializer=glorot_uniform(seed=0))(X)
    X = Dense(128, activation='relu', kernel_initializer=glorot_uniform(seed=0))(X)
    X = Dense(64, activation='relu', kernel_initializer=glorot_uniform(seed=0))(X)
    X = Dense(classes, activation='softmax', name='fc' + str(classes), kernel_initializer=glorot_uniform(seed=0))(X)
# Create model
    model = Model(inputs = X_input, outputs = X, name='ResNets50')

    return model

In [ ]:
model = ResNet50(input_shape = (128,128, 3), classes = 10)
model.compile(optimizer=Adam(learning_rate = 0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

NameError: name 'Input' is not defined

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint
import pickle

checkpoint = ModelCheckpoint(
    filepath="/content/distracted_driver_data/state-farm-distracted-driver-detection/resnet_50.weights.h5",
    monitor="val_accuracy",
    save_best_only=True,
    save_weights_only=True,
    mode="max",
    verbose=1
)

history = model.fit(
    train_scaled_data,
    y_train_label,
    batch_size=8,   # use 8 for GPU safety
    epochs=10,
    validation_data=(val_scaled_data, y_test_label),
    callbacks=[checkpoint]
)

with open("/content/distracted_driver_data/state-farm-distracted-driver-detection/resnet_50.pkl", "wb") as file:
    pickle.dump(history.history, file)

print("ResNet-50 classification model trained, and best weights saved!")

NameError: name 'model' is not defined

In [ ]:
print(ResNet50)

NameError: name 'ResNet50' is not defined